# Reading SEC Filings with `edgartools`

In this notebook we will use the [`edgartools`](https://github.com/dgunning/edgartools) Python package to:

1. Look up a public company by its ticker symbol.
2. List its recent **10-K** (annual report) filings.
3. Open the most recent 10-K and inspect its structure.
4. Extract a few named sections — _Business_, _Risk Factors_, and _Management's Discussion and Analysis_ — as clean plain text.
5. Save the extracted text to disk so the **next notebook** (`analyze-filings-with-llm.ipynb`) can feed it into an LLM.

We pin `edgartools` to the **5.x** major version so the API calls below behave the same regardless of future major releases.


## 1. Install and import

You only need to run the install cell once per environment. The version pin (`>=5.0.0,<6.0.0`) prevents an accidental jump to a future 6.x release that may rename methods.


In [28]:
# %pip install --quiet "edgartools>=5.0.0,<6.0.0"

In [29]:
import os
from pathlib import Path

import edgar
from edgar import Company, set_identity

Check `edgartools` version.


In [30]:
%pip show edgartools

Name: edgartools
Version: 5.30.0
Summary: Python library to access and analyze SEC Edgar filings, XBRL financial statements, 10-K, 10-Q, and 8-K reports
Home-page: https://github.com/dgunning/edgartools
Author: 
Author-email: Dwight Gunning <dgunning@gmail.com>
License-Expression: MIT
Location: c:\Users\ypark32\AppData\Local\miniforge3\Lib\site-packages
Requires: beautifulsoup4, httpx, httpxthrottlecache, humanize, jinja2, lxml, nest-asyncio, orjson, pandas, pyarrow, pydantic, pyrate-limiter, rank-bm25, rapidfuzz, rich, stamina, tabulate, textdistance, tqdm, truststore, unidecode
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## 2. Identify yourself to the SEC

The SEC's [fair-access policy](https://www.sec.gov/os/accessing-edgar-data) requires every automated request to include a contact email in the `User-Agent` header. `edgartools` exposes this through `set_identity()`. **Use a real email address you actually monitor** — the SEC may contact you (or rate-limit you) if your scripts misbehave.

We read the identity from an environment variable so we never hard-code a personal email into the notebook. Set `EDGAR_IDENTITY` before launching Jupyter, e.g.:

```bash
export EDGAR_IDENTITY="Jane Student jane@example.edu"
```


In [31]:
identity = os.environ.get("EDGAR_IDENTITY", "BDI 593 Student student@example.edu")
set_identity(identity)
print("SEC identity set to:", identity)

SEC identity set to: BDI 593 Student student@example.edu


## 3. Look up a company by ticker

`edgar.Company` accepts either a ticker symbol (e.g. `"AAPL"`) or a CIK (Central Index Key, the SEC's internal company ID). Tickers are easier for humans; CIKs are more stable over time (companies sometimes change tickers).

We will use **Apple Inc.** as our running example. Feel free to change `TICKER` to any large U.S. public company you are curious about.


In [32]:
TICKER = "AAPL"

company = Company(TICKER)
print(f"Name: {company.name}")
print(f"CIK : {company.cik}")

Name: Apple Inc.
CIK : 320193


## 4. List 10-K filings

Every public U.S. company files a **10-K** once per fiscal year. We can ask `edgartools` to filter the company's full filing history down to just 10-Ks.


In [33]:
tenk_filings = company.get_filings(form="10-K")
tenk_filings.head(5)

╭──────────────────────────────────────── Filings for Apple Inc. [320193] ────────────────────────────────────────╮
│                                                                                                                 │
│                                                                              Filing                             │
│    Form        Description                                                   Date         Accession Number      │
│  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
│    10-K        Annual report for public companies                            2025-10-31   0000320193-25-0000…   │
│    10-K        Annual report for public companies                            2024-11-01   0000320193-24-0001…   │
│    10-K        Annual report for public companies                            2023-11-03   0000320193-23-0001…   │
│    10-K        Annual report for public companies                     

The result is a `Filings` collection — think of it as a small DataFrame of metadata (form, filing date, accession number, primary document). Indexing it with `[0]` gives us the most recent filing as a `Filing` object.


In [34]:
latest_10k = tenk_filings[0]

print("Form           :", latest_10k.form)
print("Filed on       :", latest_10k.filing_date)
print("Period of report:", latest_10k.period_of_report)
print("Accession no.  :", latest_10k.accession_no)
print("Primary doc URL:", latest_10k.homepage_url)

Form           : 10-K
Filed on       : 2025-10-31
Period of report: 2025-09-27
Accession no.  : 0000320193-25-000079
Primary doc URL: https://www.sec.gov/Archives/edgar/data/320193/0000320193-25-000079-index.html


## 5. Parse the 10-K into a structured object

`Filing.obj()` returns a form-aware wrapper. For a 10-K, that wrapper is a `TenK` object that exposes named sections (Business, Risk Factors, MD&A, Financial Statements, …) as attributes. This is much friendlier than parsing the raw HTML yourself.


In [35]:
tenk = latest_10k.obj()
print("Object type:", type(tenk).__name__)
print("Available items:")
for item in tenk.items:
    print(" -", item)

Object type: TenK
Available items:
 - Item 1
 - Item 7A
 - Item 8
 - Item 9
 - Item 9A
 - Item 9B
 - Item 9C
 - Item 10
 - Item 11
 - Item 12
 - Item 13
 - Item 14
 - Item 15
 - Item 16
 - Item 1A
 - Item 1B
 - Item 1C
 - Item 2
 - Item 3
 - Item 4
 - Item 5
 - Item 6
 - Item 7


## 6. Extract specific sections

The three sections most useful for LLM analysis are:

- **Item 1 — Business** (`tenk.business`): a long-form description of how the company makes money.
- **Item 1A — Risk Factors** (`tenk.risk_factors`): the company's own enumeration of risks.
- **Item 7 — Management's Discussion and Analysis** (`tenk.management_discussion`): management's narrative on results.

Each accessor returns a plain Python `str` (already cleaned of HTML).


In [36]:
business_text = tenk.business or ""
risk_text = tenk.risk_factors or ""
mdna_text = tenk.management_discussion or ""

print(f"Business        : {len(business_text):>7,} characters")
print(f"Risk Factors    : {len(risk_text):>7,} characters")
print(f"MD&A            : {len(mdna_text):>7,} characters")

Business        :  16,054 characters
Risk Factors    :  68,163 characters
MD&A            :  18,018 characters


Let's preview the first ~800 characters of each section so we can sanity-check the extraction. A clean extraction should read like English prose, not like leftover HTML markup.


In [37]:
def preview(label: str, text: str, n: int = 800) -> None:
    print(f"\n=== {label} (first {n} chars) ===\n")
    print(text[:n].strip())
    print("..." if len(text) > n else "")


preview("BUSINESS", business_text)
preview("RISK FACTORS", risk_text)
preview("MD&A", mdna_text)


=== BUSINESS (first 800 chars) ===

Item 1.    Business

Company Background

The Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and sells a variety of related services. The Company’s fiscal year is the 52- or 53-week period that ends on the last Saturday of September.

Products

iPhone

iPhone® is the Company’s line of smartphones based on its iOS operating system. The iPhone line includes iPhone 17 Pro, iPhone Air™, iPhone 17, iPhone 16 and iPhone 16e.

Mac

Mac® is the Company’s line of personal computers based on its macOS® operating system. The Mac line includes laptops MacBook Air® and MacBook Pro®, as well as desktops iMac®, Mac mini®, Mac Studio® and Mac Pro®.

iPad

iPad® is the Company’s line of multipurpose tablets based on its iPadOS® operating system
...

=== RISK FACTORS (first 800 chars) ===

Item 1A.    Risk Factors

The following summarizes factors that could have a material adverse effect on the Company’s

## 7. (Optional) Peek at the financial statements

`edgartools` also parses the XBRL-tagged financial statements into pandas-friendly tables. We will not use these in the LLM notebook, but it is worth knowing they are available — they are useful for combining quantitative trends with the qualitative analysis the LLM will perform.


In [38]:
try:
    financials = tenk.financials
    income_statement = financials.income_statement()
    display(income_statement)
except Exception as exc:  # noqa: BLE001
    print("Financials not available for this filing:", exc)

                                                                                                     
                                  APPLE INC.   AAPL                                                  
                                  CONSOLIDATED STATEMENT OF INCOME                                   
                                  Sep 30, 2023 to Sep 27, 2025                                       
                                                                                                     
                                                       Sep 27, 2025   Sep 28, 2024   Sep 30, 2023    
   ───────────────────────────────────────────────────────────────────────────────────────────────   
            Net sales:                                     $416,161       $391,035       $383,285    
            Products                                       $307,003       $294,866       $298,085    
            Services                                       $109,158        $96,169

## 8. Save extracted text for the next notebook

We persist each section to `extracted/` so the LLM notebook can read it back without re-querying the SEC. Sticking to plain `.txt` files keeps the data inspectable and avoids any pickle/portability headaches.


In [39]:
out_dir = Path("extracted")
out_dir.mkdir(exist_ok=True)

meta = {
    "ticker": TICKER,
    "company_name": company.name,
    "cik": str(company.cik),
    "form": latest_10k.form,
    "accession_no": latest_10k.accession_no,
    "filing_date": str(latest_10k.filing_date),
    "period_of_report": str(latest_10k.period_of_report),
}


# Write a small metadata header at the top of each file so the source is always traceable.
def write_section(name: str, text: str) -> Path:
    path = out_dir / f"{TICKER}_{name}.txt"
    header = (
        f"# Source: SEC EDGAR {meta['accession_no']} ({meta['form']})\n"
        f"# Company: {meta['company_name']} (CIK {meta['cik']}, ticker {meta['ticker']})\n"
        f"# Filed:   {meta['filing_date']}  |  Period: {meta['period_of_report']}\n"
        f"# Section: {name}\n\n"
    )
    path.write_text(header + text, encoding="utf-8")
    return path


paths = {
    "business": write_section("business", business_text),
    "risk_factors": write_section("risk_factors", risk_text),
    "mdna": write_section("mdna", mdna_text),
}

for name, path in paths.items():
    print(f"Wrote {name:<13} -> {path}  ({path.stat().st_size:>7,} bytes)")

Wrote business      -> extracted\AAPL_business.txt  ( 16,472 bytes)
Wrote risk_factors  -> extracted\AAPL_risk_factors.txt  ( 69,105 bytes)
Wrote mdna          -> extracted\AAPL_mdna.txt  ( 18,708 bytes)


## 9. Recap

We have:

- Set the SEC identity required for polite, compliant API access.
- Looked up Apple Inc. by ticker, listed its 10-K filings, and grabbed the most recent one.
- Parsed the 10-K into a `TenK` object and extracted the _Business_, _Risk Factors_, and _MD&A_ sections as clean plain text.
- Saved each section to disk under `extracted/` for downstream use.

In the companion notebook, **`analyze-filings-with-llm.ipynb`**, we will load these `.txt` files and run a series of OpenAI-compatible chat-completion calls against them — some with free-form text outputs, some with strict JSON schemas.

:::{tip} Try this on your own
Re-run the notebook with a different `TICKER` (e.g. `"MSFT"`, `"NVDA"`, `"WMT"`) and compare how long each section is across companies. Risk Factor sections in particular vary a lot in length and tone between industries.
:::
